# TEST INFERENCE PIPELINE

In [ ]:
from pathlib import Path
import numpy as np
import torch
from torchvision.transforms import ToTensor
import io
from PIL import Image
import cv2

from torch.utils.data import DataLoader
from emma import EmmaDataset, EmmaSearcher, SearchQuery
from emma.dataloader import EmmaMapDataset
from emma.dataloader import emma_collate

from models.model import GLPDepth
from types import SimpleNamespace
from collections import OrderedDict

import utils.logging_ as logging

In [ ]:
# Check CUDA Availability

if torch.cuda.is_available():
    print("Ready")
    device = torch.device('cuda')
else:
    print("Not Ready")
    device = torch.device('cpu')

## > DATASET

In [ ]:
dataset_root = None  # e.g. Path("/path/to/EMMa")

if dataset_root is None:
    raise ValueError("Please set `dataset_root` to the path of the EMMa dataset.")

dataset_root = Path(dataset_root)

if not dataset_root.exists():
    raise FileNotFoundError(f"Dataset path does not exist: {dataset_root}")

In [ ]:
# INIT DATASET
ds = EmmaDataset(dataset_root)

In [ ]:
# QUERY
se = EmmaSearcher(ds)
query = SearchQuery(splits=["Test"], require_listed=["mass", "category"], require_present=["package_dimensions"], quantity_range=[1,1], limit=10)
asins = se.query_asins(query)
print(len(asins))

In [ ]:
to_tensor = ToTensor()
def my_transform(asin: str, rec: dict):
    image_dict = ds.get_images(rec)

    first_img_key, first_img_bytes = next(iter(image_dict.items()))
    first_img = Image.open(io.BytesIO(first_img_bytes)).convert("RGB")
    first_img = np.array(first_img)
    first_img = resize_image(first_img)

    dimensions = np.squeeze(ds.get_package_dimensions(rec)) * 2.54 / 100.0
    normalization = np.float32(np.linalg.norm(dimensions))

    return {
        "image": to_tensor(first_img),
        "normalization": normalization,
        "asin": asin,
    }

In [ ]:
batch_size = 4

loader = DataLoader(
    EmmaMapDataset(ds, asins, transform=my_transform),
    batch_size=batch_size, shuffle=False, collate_fn=emma_collate,
)

## > MODEL

In [ ]:
glp_ckpt = None  # e.g. Path("/path/to/GLP-CKPT")

if glp_ckpt is None:
    raise ValueError("Please set `glp_ckpt` to the path of the GLP-Depth checkpoint.")

glp_ckpt = Path(glp_ckpt)

if not glp_ckpt.exists():
    raise FileNotFoundError(f"Checkpoint path does not exist: {glp_ckpt}")

In [ ]:
# Model Arguments
args = SimpleNamespace(min_depth_eval=1e-3, max_depth_eval=2.63, max_depth=2.63)

In [ ]:
model_weight = torch.load(glp_ckpt)
if 'module' in next(iter(model_weight.items()))[0]:
    model_weight = OrderedDict((k[7:], v) for k, v in model_weight.items())

model = GLPDepth(max_depth=args.max_depth, is_train=False).to(device)
model.load_state_dict(model_weight)
model.eval()

## > AUGMENTATION

In [ ]:
def clamp_pred(args, pred):
    pred = torch.nan_to_num(pred, nan=args.min_depth_eval, posinf=args.max_depth_eval, neginf=args.max_depth_eval)
    return pred.clamp_min(0.0)

def resize_image(image):
    border_size = (640 - 480) // 2
    border_color = [255, 255, 255]
    image = cv2.resize(image, (480, 480))
    image = cv2.copyMakeBorder(image, 0, 0, border_size, border_size, cv2.BORDER_CONSTANT, value=border_color)

    return image

def pad_to_batch_size(batch_tensor: torch.Tensor, batch_size: int) -> torch.Tensor:
    current_size = batch_tensor.shape[0]
    if current_size == batch_size:
        return batch_tensor

    pad_count = batch_size - current_size
    padding = batch_tensor[-1:].expand(pad_count, *batch_tensor.shape[1:])
    return torch.cat((batch_tensor, padding), dim=0)

In [ ]:
inference_results = []

scale = 1000.0
for batch_idx, batch in enumerate(loader):
    input_RGBs = batch['image'].to(device)
    normalizations = batch['normalization'].to(device)
    asin_batch = list(batch["asin"])
    real_batch_size = len(asin_batch)

    with torch.no_grad():
        pred = model(pad_to_batch_size(input_RGBs, batch_size))
    pred_d = pred['pred_d'].squeeze()[:real_batch_size]

    for asin, depth_map, normalization in zip(asin_batch, pred_d, normalizations, strict=True):
        depth_map = clamp_pred(args, depth_map * normalization)
        depth_map_img = (depth_map.cpu().numpy() * scale).astype(np.uint16)
        
        inference_results.append({
            "depth": depth_map_img,
            "scale": scale,
        })

    logging.progress_bar(batch_idx, len(loader), 1, 1)
        
print("Done")

## > VISUALIZE

In [ ]:
for result in inference_results[:10]:
    pred_d_vis = result["depth"] / result["scale"]   
    pred_d_vis = (pred_d_vis - pred_d_vis.min()) / (pred_d_vis.max() - pred_d_vis.min() + 1e-8)
    pred_d_vis = (pred_d_vis * 255).astype(np.uint8)


    print("DEPTH MIN: ", pred_d.min())
    print("DEPTH MAX: ", pred_d.max())
    print("NORMALIZATION: ", normalization)
    Image.fromarray(pred_d_vis).show()